# Transformer Architecture

**Module:** 05 — LLM Fundamentals

The high-level Transformer stack, encoder vs decoder designs, and positional information.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Describe embeddings → blocks → LM head
- Contrast encoder-only, decoder-only, and encoder-decoder
- Explain why positional information is required
- Relate architecture choices to LLM products you use


## High-Level Stack

**Definition.** A Transformer LM maps tokens to vectors, applies N blocks of attention+MLP, then predicts vocab logits.

**Why it matters.** This stack is the computational backbone of nearly all modern LLMs.

**How it works.** Token embed (+ position) → Pre-Norm → Multi-head attention → MLP → residual links → LM head.

**Intuition.** A assembly line: mix information across positions, then rewrite features.

**Common pitfalls.**
- Ignoring residual/normalization details when debugging NaNs
- Confusing block count with 'smartness' alone

**When to use.** Reading papers/model cards; sizing inference.

```mermaid
flowchart TB
  Tok[Token ids] --> Emb[Embedding + position]
  Emb --> B1[Block x N]
  B1 --> Head[LM head]
  Head --> Logits[Vocab logits]
```


In [ ]:
# Demo 1 — shapes through a toy stack
B, T, D, V = 2, 5, 16, 100  # batch, time, dim, vocab
shapes = {
  "tokens": (B, T),
  "embed": (B, T, D),
  "attn_out": (B, T, D),
  "mlp_out": (B, T, D),
  "logits": (B, T, V),
}
for k, v in shapes.items():
    print(f"{k:10s} {v}")


In [ ]:
# Demo 2 — parameter ballpark for one block
def block_params(d, ff=4):
    # attn qkv+out ~ 4*d*d; mlp ~ 2*d*(ff*d)
    return 4*d*d + 2*d*(ff*d)
for d in [768, 4096]:
    print(d, "one_block~", block_params(d)/1e6, "M")


In [ ]:
# Demo 3 — residual stream intuition
import numpy as np
x = np.zeros(8); delta = np.arange(8)
y = x + delta  # residual add
print(y)


### Try it yourself — High-Level Stack

1. Draw the stack and label where KV cache lives at inference.


## Encoder vs Decoder

**Definition.** **Encoders** bidirectional-encode sequences; **decoders** attend causally for generation; **encoder-decoder** (T5-style) encode input then decode output.

**Why it matters.** Task type dictates architecture: understanding vs generation vs transduction.

**How it works.** LLMs for chat are usually decoder-only; embeddings/BERT-like are encoder-only.

**Intuition.** Encoder reads with full context; decoder writes left-to-right.

**Common pitfalls.**
- Using encoder-only models to generate long text
- Forgetting causal masks in DIY decoders

**When to use.** Pick encoder for embeddings/classification; decoder for generative LLMs.

| Type | Attention | Typical use |
|------|-----------|-------------|
| Encoder | Bidirectional | Embeddings, NLU |
| Decoder | Causal | LLMs, completion |
| Enc-Dec | Mix | Seq2seq classic |


In [ ]:
# Demo 1 — causal mask
import numpy as np
T = 5
mask = np.tril(np.ones((T, T)))
print(mask)


In [ ]:
# Demo 2 — architecture chooser
def choose(task):
    return {
      "generate_chat": "decoder-only",
      "embed_docs": "encoder-only",
      "translate": "encoder-decoder or decoder-only",
    }[task]
for t in ["generate_chat", "embed_docs", "translate"]:
    print(t, "→", choose(t))


In [ ]:
# Demo 3 — bidirectional vs causal attention counts
T=4
print("encoder edges", T*T)
print("decoder causal edges", T*(T+1)//2)


### Try it yourself — Encoder vs Decoder

1. Why are most chat LLMs decoder-only today?


## Positional Information

**Definition.** Attention is permutation-capable without positions; **positional encodings/RoPE/ALiBi** inject order.

**Why it matters.** Language is ordered—'dog bites man' ≠ 'man bites dog'.

**How it works.** Add absolute encodings or apply rotary/relative biases inside attention scores.

**Intuition.** Give each token a sense of where it sits in line.

**Common pitfalls.**
- Extrapolating far beyond trained context without care
- Mixing incompatible position schemes

**When to use.** All Transformer LMs—check model card for RoPE scaling notes.


In [ ]:
# Demo 1 — sin/cos absolute position features
import numpy as np
def sinusoidal(T, D):
    pos = np.arange(T)[:, None]
    i = np.arange(D)[None, :]
    angles = pos / (10000 ** (2*(i//2)/D))
    pe = np.zeros((T, D))
    pe[:, 0::2] = np.sin(angles[:, 0::2])
    pe[:, 1::2] = np.cos(angles[:, 1::2])
    return pe
print(sinusoidal(4, 8).round(2))


In [ ]:
# Demo 2 — RoPE intuition: rotate q/k by position
import numpy as np
theta = 0.5
R = np.array([[np.cos(theta), -np.sin(theta)],[np.sin(theta), np.cos(theta)]])
q = np.array([1.0, 0.0])
print(R @ q)


In [ ]:
# Demo 3 — length extrapolation flag
train_ctx, serve_ctx = 8192, 128000
print({"needs_scaling_strategy": serve_ctx > train_ctx})


### Try it yourself — Positional Information

1. Compare absolute sinusoidal vs RoPE in one sentence each.


## Glossary

- **causal mask**: Prevents attending to future tokens
- **RoPE**: Rotary positional embeddings


### Workshop drill — Transformer Architecture (1)

Restate each section heading as a single exam-ready sentence.


In [ ]:
# Workshop drill 1 — Transformer Architecture
headings = ['High-Level Stack', 'Encoder vs Decoder', 'Positional Information']
for h in headings:
    print('-', h, '→', '...')


### Workshop drill — Transformer Architecture (2)

Change one hyperparameter/assumption in a demo and predict the effect before running.


In [ ]:
# Workshop drill 2 — Transformer Architecture
print('prediction: ...')
print('observation: ...')
print('delta: ...')


### Workshop drill — Transformer Architecture (3)

List production risks (cost, latency, safety, quality) for this topic.


In [ ]:
# Workshop drill 3 — Transformer Architecture
for r in ['cost','latency','safety','quality']:
    print(f'{r}:')


### Workshop drill — Transformer Architecture (4)

Write a tiny unit-testable helper related to the lesson and assert two cases.


In [ ]:
# Workshop drill 4 — Transformer Architecture
def ok(x):
    return x is not None
assert ok(1) and not ok(None)
print('ok')


### Workshop drill — Transformer Architecture (5)

Sketch an API request/response JSON for a realistic call tied to this topic.


In [ ]:
# Workshop drill 5 — Transformer Architecture
import json
print(json.dumps({'model':'...','input':'...','output':'...'}, indent=2))


### Workshop drill — Transformer Architecture (6)

Compare two design alternatives in a markdown table (fill TODOs).


In [ ]:
# Workshop drill 6 — Transformer Architecture
print('| option | pros | cons |')
print('|--------|------|------|')
print('| A | TODO | TODO |')
print('| B | TODO | TODO |')


## Summary & Key Takeaways

- Transformers stack attention+MLP blocks over a residual stream
- Decoder-only causal models dominate generative LLMs
- Positional schemes encode order and affect long-context behavior

### Practice

Read one model card and identify: blocks, hidden size, context length, position type.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
